In [17]:
from abc import ABC, abstractmethod
from collections import namedtuple

Customer = namedtuple('Customer', 'name fidelity')

In [18]:
park = Customer('Park', 100)
park

Customer(name='Park', fidelity=100)

Customer(name='Park', fidelity=100)

In [19]:
class LineItem:
    """구매할 물품/갯수 생성하여 총 가격 반환"""
    def __init__(self,product,quantity,price):
        self.product = product
        self.quantity = quantity
        self.price = price
    
    def total(self):
        return self.price * self.quantity

In [20]:
class Order:  # Context
    """고객(customer)의 LineItem 클래스의 인스턴스들을 cart로 받아서 총 계산할 가격을 산출"""
    def __init__(self, customer, cart, promotion=None):
        self.customer = customer
        self.cart = list(cart)
        self.promotion = promotion  # 할인 객체

    def total(self):
        """_total 속성이 없으면 전체 계산할 값을 계산"""
        if not hasattr(self, '_total'):
            self._total = sum(item.total() for item in self.cart)
        return self._total

    def due(self):
        """할인금액 차감"""
        if self.promotion is None:
            discount = 0
        else:
            discount = self.promotion.discount(self)  # self = Order 객체
        return self.total() - discount

    def repr_(self):
        fmt = '<Order total: {:.2f} due: {:.2f}'
        return fmt.format(self.total(), self.due())


In [21]:
class Promotion(ABC): #strategy : abstract base class
    """할인 혜택 클래스들의 형태 선언"""
    
    @abstractmethod #이 클래스를 상속하는 클래스는 반드시 이 메서드 선언 필요
    def discount(self,order):
        """할인액을 구체적인 숫자로 반환"""
        pass

In [22]:
class FidelityPromo(Promotion):
    """충성도 점수가 1000점 이상인 고객에게 전체 5% 할인 적용"""

    def discount(self, order):
        return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0


class BulkItemPromo(Promotion):
    """20개 이상의 동일 상품을 구입하면 10% 할인 적용"""

    def discount(self, order):
        discount = 0
        for item in order.cart:
            if item.quantity >= 20:
                discount += item.total() * 0.1
        return discount


class LargeOrderPromo(Promotion):
    """10종류 이상의 상품을 구입하면 전체 7% 할인 적용"""

    def discount(self, order):
        distinct_items = {item.product for item in order.cart}
        if len(distinct_items) >= 10:
            return order.total() * 0.07
        return 0

In [23]:
joe = Customer('Jone Doe', 0)
ann = Customer('Ann Smith', 1100)
cart = [LineItem('banana',4,.5),
        LineItem('apple',10,1.5),
        LineItem('watermellon',5,5.0)]

In [24]:
Order(joe, cart, FidelityPromo())

In [25]:
Order(ann, cart, FidelityPromo())

In [26]:
banana_cart = [LineItem('banana',30,.5),
               LineItem('apple',10,1.5)]

In [27]:
Order(joe, banana_cart, BulkItemPromo())

In [28]:
long_order = [LineItem(str(item_code),1,1.0)
               for item_code in range(10)] #물품 종류 10개

In [29]:
Order(joe, long_order, LargeOrderPromo())

In [30]:
Order(joe,cart,LargeOrderPromo())

In [31]:
class Order:  # Context
    """고객(namedtuple) 및 LineItem 클래스의 인스턴스들을 cart로 받아서 총 계산할 가격을 산출"""

    def __init__(self, customer, cart, promotion=None):
        self.customer = customer
        self.cart = list(cart)
        self.promotion = promotion  # 할인 객체

    def total(self):
        if not hasattr(self, "__total"):
            self.__total = sum(item.total() for item in self.cart)
        return self.__total

    def due(self):
        if self.promotion is None:
            discount = 0
        else:
            discount = self.promotion(self)  # promotion은 함수 객체
        return self.total() - discount

    def __repr__(self):
        fmt = "<Order total: {:.2f} due: {:.2f}>"
        return fmt.format(self.total(), self.due())

In [32]:
def fidelity_promo(order):
    """충성도 점수가 1000점 이상인 고객에게 전체 5% 할인 적용"""
    return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0


def bulk_item_promo(order):
    """20개 이상의 동일 상품을 구입하면 10% 할인 적용"""
    discount = 0
    for item in order.cart:
        if item.quantity >= 20:
            discount += item.total() * 0.1
    return discount


def large_order_promo(order):
    """10종류 이상의 상품을 구입하면 전체 7% 할인 적용"""
    distinct_items = {item.product for item in order.cart}
    if len(distinct_items) >= 10:
        return order.total() * 0.07
    return 0

In [33]:
Order(joe, cart, fidelity_promo)

<Order total: 42.00 due: 42.00>

In [34]:
Order(ann, cart, fidelity_promo)

<Order total: 42.00 due: 39.90>

In [35]:
Order(joe, banana_cart,fidelity_promo)

<Order total: 30.00 due: 30.00>

In [36]:
Order(ann, long_order, large_order_promo)

<Order total: 10.00 due: 9.30>

In [37]:
promos = [fidelity_promo, bulk_item_promo, large_order_promo]
#함수들로 구현된 전략들의 리스트

def best_promo(order):
    """최대로 할인받을 금액을 반환"""
    return max(promo(order) for promo in promos)

In [38]:
Order(joe, long_order, best_promo)

<Order total: 10.00 due: 9.30>

In [39]:
Order(joe, banana_cart, best_promo)

<Order total: 30.00 due: 28.50>

In [40]:
Order(ann, cart, best_promo)

<Order total: 42.00 due: 39.90>

In [41]:
globals().keys()

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__vsc_ipynb_file__', '_i', '_ii', '_iii', '_i1', 'ABC', 'abstractmethod', 'namedtuple', 'Customer', '_i2', '_i3', 'park', '_3', '_i4', 'LineItem', '_i5', 'Order', '_i6', 'Promotion', '_i7', 'FidelityPromo', 'BulkItemPromo', 'LargeOrderPromo', '_i8', 'joe', 'ann', '_i9', '_i10', '_i11', '_11', '_i12', '_i13', '_i14', '_i15', '_i16', '_i17', '_i18', '_18', '_i19', '_i20', '_i21', '_i22', '_i23', 'cart', '_i24', '_24', '_i25', '_25', '_i26', 'banana_cart', '_i27', '_27', '_i28', 'long_order', '_i29', '_29', '_i30', '_30', '_i31', '_i32', 'fidelity_promo', 'bulk_item_promo', 'large_order_promo', '_i33', '_33', '_i34', '_34', '_i35', '_35', '_i36', '_36', '_i37', 'promos', 'best_promo', '_i38', '_38', '_i39', '_39', '_i40', '_40', '_i41'])

In [42]:
globals()['best_promo']

<function __main__.best_promo(order)>

In [43]:
promos = [globals()[name] for name in globals()
          if name.endswith('_promo')
          and name != 'best_promo'] #재귀 방지
promos

[<function __main__.fidelity_promo(order)>,
 <function __main__.bulk_item_promo(order)>,
 <function __main__.large_order_promo(order)>]